In [0]:
from pyspark.sql.types import StructField, StructType, IntegerType,StringType
from pyspark.sql.functions import concat,coalesce, col, lit, concat_ws, when, countDistinct

Creating Input dataframe

In [0]:
schema = StructType([
    StructField("id",IntegerType(),True),
    StructField("first name",StringType(),True),
    StructField("last name",StringType(),True),
    StructField("Expected DOJ",StringType(),True),
    StructField("Department",StringType(),True),
    StructField("Current CTC",IntegerType(),True)
])
data = [
    (1,'Amit','Sharma','7/15/2023','Engineering',1250000),
    (2,'Priya','Mehta','5/20/2022','IT',980000),
    (3,'Rajesh','Verma','1/10/2024','Engineering',1500000),
    (4,'Neha',None,'9/25/2023','IT',1320000),
    (5,'Rohit','Malhotra','11/30/2022','Engineering',1070000)
]
df = spark.createDataFrame(data,schema)
df.display()

In [0]:
# df.select(countDistinct("department")).show()

Generate Name using first name and last name

In [0]:
# 1: using concat where we specify space seperately
df_1 = df.withColumn('Full_name', concat(coalesce(col('first name'),lit(''))
                                         ,lit(' ')
                                         ,coalesce(col('last name'),lit(''))
                                         ))
df_1.display()

# 2: Using concat_ws this excludes null without using coalesce

df = df.withColumn("full_name",concat_ws(" ",col('first name'),
                                           col('last name'),lit('')
                                           ))
df.display()                                          

Offer a CTC for Engineering department 25% and IT 17%

In [0]:
df= df.withColumn("Offered CTC", when(col("Department") == 'Engineering', col("Current CTC")*.25 + col("Current CTC"))
                                    .when(col("Department") == 'IT', col("Current CTC")*.17+col("Current CTC"))
                                    .otherwise(col("Current CTC")).cast(IntegerType()))
df.display()

In [0]:
df_final = df.select('id','full_name','Expected DOJ','Department','Current CTC','Offered CTC')
df_final.display()